# How long does a moving wall take to set a fluid layer in motion?

Start the top of a narrow gap suddenly and follow the velocity, volume flow and wall stresses as viscous diffusion builds a Couette profile.

## Remove the final state before separating variables

Take two large, parallel plates separated by $H$, with $y=0$ at the fixed plate. The liquid starts from rest. At $t=0$ the upper plate begins to move at constant speed $U$. For an incompressible Newtonian liquid with no imposed pressure gradient, the unidirectional velocity $u(y,t)$ has zero convective acceleration. The streamwise momentum balance on a thin slice then reduces to

$$
\rho\,\partial_tu=\partial_y(\mu\,\partial_yu).
$$

With constant viscosity, kinematic viscosity $\nu=\mu/\rho$ is the momentum diffusivity.

Use $\eta=y/H$ and $F=\nu t/H^2$. The linear profile $u/U=\eta$ solves the eventual steady equation and both wall conditions. Subtract it by setting $w=u/U-\eta$. Both boundaries now require $w=0$, which selects eigenfunctions $\sin(n\pi\eta)$ for $w_F=w_{\eta\eta}$. Each amplitude decays as $\exp(-n^2\pi^2 F)$. The interior initially has $w=-\eta$. Multiplying by a sine and integrating from zero to one gives

$$
b_n=-2\int_0^1\eta\sin(n\pi\eta)\,d\eta=2(-1)^n/(n\pi).
$$

Combining the parts gives

$$\frac{u}{U}=\eta+\frac{2}{\pi}\sum_{n=1}^{\infty}\frac{(-1)^n}{n}\sin(n\pi\eta)e^{-n^2\pi^2 F}.$$

For any positive time, higher modes decay rapidly. At the initial top corner the sudden change in surface velocity conflicts with the initially resting liquid. The mathematical wall gradient there diverges as $t\downarrow0$. A finite starting acceleration would regularize that prediction. Fluid well away from that wall sees little change until the diffusion distance, of order $\sqrt{\nu t}$, becomes comparable with its distance to the moving surface.

## Distinguish stored momentum from transmitted shear

Integrating the profile over the gap eliminates the even sine modes. The dimensionless volume flow per unit span becomes

$$
q/(UH)=1/2-(4/\pi^2)\sum_{n\ {\rm odd}} e^{-n^2\pi^2 F}/n^2.
$$

Differentiating at the walls gives

$$
\tau_\mathrm{top}/(\mu U/H)=1+2\sum_n e^{-n^2\pi^2F},
$$

$$
\tau_\mathrm{bottom}/(\mu U/H)=1+2\sum_n(-1)^n e^{-n^2\pi^2 F}.
$$

Here both symbols report $\mu u_y$ evaluated in the positive $y$ direction. The forces on the fluid act in opposite directions, so the net streamwise force per unit plate area is $\tau_\mathrm{top}-\tau_\mathrm{bottom}$.

Set $H=4$ mm, $\nu=1.6\times10^{-5}$ m$^2$/s, $\rho=900$ kg/m$^3$ and $U=0.20$ m/s. The gap diffusion time is 1 s. At 0.1 s the flow is $0.34894 UH$, already 69.8% of its final value. The wall gradients still differ sharply, measuring 1.7843 and 0.2929 times the eventual shear of 0.72 Pa. Integrating the original momentum equation over the layer verifies that their difference accelerates the stored streamwise momentum, $\rho\,dq/dt$.

## Set the experimental inputs

Edit these values, then run the calculation and both figures again.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

teal, orange, blue = "#19596a", "#aa462a", "#3868a0"
def axes(xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    ax.grid(alpha=0.2)
    return fig, ax

In [ ]:
gap_m = 0.004
speed_m_s = 0.20
viscosity_m2_s = 1.6e-5
density_kg_m3 = 900.0
observe_s = 0.10

In [ ]:
assert min(gap_m, speed_m_s, viscosity_m2_s, density_kg_m3, observe_s) > 0
diffusion_s = gap_m**2 / viscosity_m2_s
fourier_time = observe_s/diffusion_s
assert fourier_time >= 0.0005, "Increase the observation time to resolve the startup with 100 modes."
modes = np.arange(1, 101)
odd = modes[modes % 2 == 1]
def profile(eta, F):
    amplitudes = 2*(-1.0)**modes*np.exp(-(modes*np.pi)**2*F)/(np.pi*modes)
    return eta + np.sin(np.outer(eta, modes*np.pi)) @ amplitudes

def scaled_flow(F):
    return 0.5 - 4/np.pi**2*np.sum(np.exp(-(odd*np.pi)**2*F)/odd**2)

def scaled_shear(F):
    decay = np.exp(-(modes*np.pi)**2 * F)
    return 1+2*decay.sum(), 1+2*((-1.0)**modes*decay).sum()

top, bottom = scaled_shear(fourier_time)
flow = scaled_flow(fourier_time)
steady_pa = density_kg_m3*viscosity_m2_s*speed_m_s/gap_m
derivative = 4*np.exp(-(odd*np.pi)**2*fourier_time).sum()
assert np.isclose(derivative, top-bottom)
print(f"Diffusion time: {diffusion_s:.3g} s. Flow per width: {flow*speed_m_s*gap_m:.6g} m²/s.")
print(f"Top and bottom shear: {top*steady_pa:.4g} Pa, {bottom*steady_pa:.4g} Pa")

In [ ]:
eta = np.linspace(0, 1, 160)
fig, ax = axes("Streamwise velocity (m/s)", "Height above fixed wall (mm)")
for F, label, color in [(0.02, "Early", blue), (fourier_time, "Selected", orange), (0.40, "Late", teal)]:
    ax.plot(speed_m_s*profile(eta, F), 1000*gap_m*eta, color=color, label=f"{label}: {F*diffusion_s:.3g} s")
ax.plot(speed_m_s*eta, 1000*gap_m*eta, "k--", label="Steady flow")
ax.legend()
plt.show()

The top wall fixes the velocity at one end immediately. The curvature in the interior tracks how momentum is still entering the liquid.

In [ ]:
times = np.geomspace(0.006, 1.2, 180)
stresses = steady_pa*np.array([scaled_shear(F) for F in times])
fig, ax = axes("Time after wall startup (s)", "Wall shear (Pa)")
ax.semilogx(times*diffusion_s, stresses[:,0], color=orange, label="Moving wall")
ax.semilogx(times*diffusion_s, stresses[:,1], color=teal, label="Fixed wall")
ax.axhline(steady_pa, color="black", linestyle="--", label="Steady value")
ax.legend()
plt.show()

Initially the driving wall accelerates the upper layer. The fixed wall feels an appreciable force later, even as the force required at the driving wall declines.

## Problem 1 · Predict the flow-settling time

Estimate when the flow reaches 99% of its final value. Why is one mode enough for this estimate?

Divide the missing flow by its final value to obtain $(8/\pi^2)\sum_{n\ {\rm odd}}e^{-n^2\pi^2 F}/n^2$. Its leading term reaches 0.01 at $F=\ln[8/(0.01\pi^2)]/\pi^2=0.44532$, or 0.44532 s with the given gap. At this time the next contributing mode contains $e^{-9\pi^2F}/9$, below $10^{-17}$. At much earlier times several modes reconstruct the thin moving layer.

## Problem 2 · Double the gap

The plate speed and liquid stay fixed while $H$ doubles. Compare the 99% settling time, steady shear, and steady flow per unit width in the two gaps.

The dimensionless solution reaches the same settling criterion at the same $F$. Since $t=FH^2/\nu$, the wider gap takes four times as long, about 1.78 s. The steady gradient is $U/H$, so its shear halves to 0.36 Pa. The steady flow is the area under a triangle of height $U$ and width $H$, namely $UH/2$. It doubles to $8.0\times10^{-4}$ m$^2$/s.